# Sweep 16 — Champion × DDI Alpha: Official Thesis Run

**Architecture locked from Sweeps 11a–15b (660 total runs).**
**DDI alpha variants added from Sweep 17 Gap C discovery.**

| Component | Champion | Locked by |
|-----------|----------|-----------|
| Temporal encoder | `imdr_infused` | Sweep 14a |
| Graph layer | `gcn` | Sweep 14b |
| Decoder | `heidr` | Sweep 14c |
| Loss | L4_jac_heavy (bce=0.3, jac=1.5, m=0.05) | Sweep 15a |
| Graph encoder | `drug_gnn` | Sweep 15b |
| Fusion | `film` | Sweep 12b |
| Aggregator | `last`, ar_seq=20 | Sweep 13a |
| Note projection | `note_proj_dim=128` | Sweep 12a |
| Lab features | 200 labs → 400d | Sweep 11b |
| Modality stack | notes + labs + copy | Sweep 11a |

## DDI Alpha Variants (Sweep 17 Gap C)

| Cell | alpha | Jaccard (est.) | DDI Rate (est.) | Profile |
|------|-------|---------------|-----------------|---------|
| 5 | **0.0** | ~0.568 | ~0.082 | Max performance |
| 6 | **0.2** | ~0.564 | ~0.069 | Balanced ← recommended |
| 7 | **0.5** | ~0.550 | ~0.047 ✅ | DDI target met |

**Total: 15 runs (3 alphas × 5 seeds). Each cell ~5h on Kaggle GPU.**
Skip-resume: re-running a cell safely skips completed seeds.

In [ ]:
import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
!pip install -q torch_geometric
!pip install -q pyyaml pandas numpy scikit-learn


In [ ]:
import os, sys, glob

if os.path.exists("/kaggle"):
    print("Running on Kaggle")
    os.chdir("/kaggle/working")
    os.system("rm -rf ./data ./src")
    os.makedirs("data/processed", exist_ok=True)
    os.makedirs("data/embeddings", exist_ok=True)

    train_paths = glob.glob("/kaggle/input/**/train.py", recursive=True)
    if not train_paths:
        raise FileNotFoundError("train.py not found in /kaggle/input")
    src_dir = os.path.dirname(train_paths[0])
    os.system(f"cp -r {src_dir} /kaggle/working/src")
    sys.path.append("/kaggle/working/src")

    for find_file in ["cohort_mimic3.pkl", "records_final.pkl"]:
        found = glob.glob(f"/kaggle/input/**/{find_file}", recursive=True)
        if found:
            processed_dir = os.path.dirname(found[0])
            print(f"Found processed dir at: {processed_dir}")
            for root, dirs, files in os.walk(processed_dir):
                rel = os.path.relpath(root, processed_dir)
                target_dir = os.path.join("./data/processed", rel) if rel != "." else "./data/processed"
                os.makedirs(target_dir, exist_ok=True)
                for fname in files:
                    src = os.path.join(root, fname)
                    link = os.path.join(target_dir, fname)
                    if not os.path.exists(link):
                        try:
                            os.symlink(src, link)
                        except OSError:
                            os.system(f"cp '{src}' '{link}'")
            break

    emb_paths = glob.glob("/kaggle/input/**/code_embeddings.pt", recursive=True)
    if not emb_paths:
        raise FileNotFoundError("code_embeddings.pt not found")
    emb_dir = os.path.dirname(emb_paths[0])
    for fpath in glob.glob(f"{emb_dir}/*"):
        fname = os.path.basename(fpath)
        link = f"./data/embeddings/{fname}"
        if not os.path.exists(link):
            os.symlink(fpath, link)

    note_paths = glob.glob("/kaggle/input/**/note_embeddings_mimic3.pkl", recursive=True)
    if note_paths:
        note_src = note_paths[0]
        note_link = "./data/processed/note_embeddings_mimic3.pkl"
        if not os.path.exists(note_link):
            os.symlink(note_src, note_link)
        print(f"Note embeddings linked: {note_src}")
    else:
        print("WARNING: note_embeddings_mimic3.pkl not found")

    # Patch 1: registry.py
    reg_path = "/kaggle/working/src/model/registry.py"
    if os.path.exists(reg_path):
        with open(reg_path, "r") as rf:
            content = rf.read()
        if "import inspect" not in content:
            old = "        return cls(**kwargs)"
            new = (
                "        import inspect\n"
                "        sig = inspect.signature(cls.__init__)\n"
                "        has_var_keyword = any(p.kind == inspect.Parameter.VAR_KEYWORD "
                "for p in sig.parameters.values())\n"
                "        if has_var_keyword:\n"
                "            filtered_kwargs = kwargs\n"
                "        else:\n"
                "            filtered_kwargs = {k: v for k, v in kwargs.items() if k in sig.parameters}\n"
                "        return cls(**filtered_kwargs)"
            )
            if old in content:
                content = content.replace(old, new)
                with open(reg_path, "w") as wf:
                    wf.write(content)
                print("  registry.py patched")

    # Patch 2: drug_gnn.py
    gnn_path = "/kaggle/working/src/model/graph_encoders/drug_gnn.py"
    if os.path.exists(gnn_path):
        with open(gnn_path, "r") as rf:
            content = rf.read()
        if "ehr_adj" not in content:
            old = "        use_lab_nodes: bool = False,\n        lab_embeddings: torch.Tensor | None = None,   # (num_labs, 768)\n    ):"
            new = "        use_lab_nodes: bool = False,\n        lab_embeddings: torch.Tensor | None = None,   # (num_labs, 768)\n        ehr_adj=None,\n        **kwargs,\n    ):"
            if old in content:
                content = content.replace(old, new)
                with open(gnn_path, "w") as wf:
                    wf.write(content)
                print("  drug_gnn.py patched")

    # Patch 3: dcma_decoder.py
    dcma_path = "/kaggle/working/src/model/decoders/dcma_decoder.py"
    if os.path.exists(dcma_path):
        with open(dcma_path, "r") as rf:
            content = rf.read()
        if "note_proj" not in content:
            content = content.replace(
                "    def __init__(self, hidden_dim: int = 256, dropout: float = 0.3, **kwargs):\n"
                "        super().__init__()\n"
                "        self.attn = DrugModalityAttention(hidden_dim, dropout)",
                "    def __init__(self, hidden_dim: int = 256, dropout: float = 0.3,\n"
                "                 note_repr_dim: int = 768, lab_repr_dim: int = 400, **kwargs):\n"
                "        super().__init__()\n"
                "        self.attn = DrugModalityAttention(hidden_dim, dropout)\n"
                "        self.hidden_dim = hidden_dim\n"
                "        self.note_proj = nn.Linear(note_repr_dim, hidden_dim)\n"
                "        self.lab_proj  = nn.Linear(lab_repr_dim,  hidden_dim)",
            )
            content = content.replace(
                "            tokens.append(F.normalize(notes_repr, dim=-1).unsqueeze(1))",
                "            tokens.append(self.note_proj(F.normalize(notes_repr, dim=-1)).unsqueeze(1))",
            )
            content = content.replace(
                "            tokens.append(F.normalize(labs_repr, dim=-1).unsqueeze(1))",
                "            tokens.append(self.lab_proj(F.normalize(labs_repr, dim=-1)).unsqueeze(1))",
            )
            with open(dcma_path, "w") as wf:
                wf.write(content)
            print("  dcma_decoder.py patched")
        else:
            print("  dcma_decoder.py already patched")

print("Setup done:", os.getcwd())


In [ ]:
import yaml, subprocess, gc, json, numpy as np
from pathlib import Path

BASE_CONFIG = "src/config.yaml"

def make_config(output_path, model_overrides=None, training_overrides=None,
                preprocessing_overrides=None, smoke_epochs=None):
    with open(BASE_CONFIG, "r") as f:
        cfg = yaml.safe_load(f)
    if model_overrides:
        for k, v in model_overrides.items():
            cfg["model"][k] = v
    if training_overrides:
        for k, v in training_overrides.items():
            cfg["training"][k] = v
    if preprocessing_overrides:
        for k, v in preprocessing_overrides.items():
            cfg["preprocessing"][k] = v
    if smoke_epochs is not None:
        cfg["training"]["epochs"] = smoke_epochs
    with open(output_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    return str(output_path)

# ── Champion architecture locked from sweeps 11a–15b ─────────────────────────
CHAMPION_MODEL = {
    "hidden_dim":         128,          # [17D] pure win: +0.0013 Jac, 3× fewer params
    "ar_max_seq_len":     20,           # [13a]
    "note_proj_dim":      64,           # [12a] = hidden_dim // 2
    "graph_encoder_type": "drug_gnn",   # [15b]
    "graph_layer_type":   "gcn",        # [14b]
    "hgt_layers":         2,
    "encoder_type":       "imdr_infused",  # [14a]
    "predictor_type":     "heidr",      # [14c]
}
CHAMPION_TRAINING = {
    "bce_weight":           0.3,        # [15a] L4_jac_heavy
    "soft_jaccard_weight":  1.5,
    "margin_weight":        0.05,
}
BACKBONE_FLAGS = (
    "--device cuda "
    "--visit_level_training "
    "--fusion_strategy film "
    "--aggregator_type last "
)
LAB_FLAGS = "--lab_pkl data/processed/lab_vectors_200labs.pkl --num_labs 200"
CHAMPION_PREP = {"lab_dim": 400}

SEEDS     = [42, 123, 456, 789, 1024]
# DDI alpha variants (Sweep 17 Gap C finding):
#   0.0  — no DDI loss, max Jaccard                   (~0.568)
#   0.2  — balanced: -0.004 Jac, DDI rate -16%        (~0.564 / DDI~0.069)
#   0.5  — safety-first: DDI <0.06 target, -0.018 Jac (~0.550 / DDI~0.047)
DDI_ALPHAS = [0.0, 0.2, 0.5]

reports_dir = Path("experiment_reports/sweep_16_champion")
results_log = []

def alpha_tag(a):
    return str(a).replace(".", "p")

def run_champion(ddi_alpha):
    # Run 5 seeds for the champion config at the given ddi_alpha.
    # Skip-aware: checks new name (champion_full_alpha{a}_seed{s}) AND
    # legacy name (champion_full_seed{s}) for alpha=0.0 backward-compat.
    import torch
    atag   = alpha_tag(ddi_alpha)
    cfgp   = f"s16_champion_alpha{atag}.yaml"
    make_config(cfgp,
                model_overrides=CHAMPION_MODEL,
                training_overrides=CHAMPION_TRAINING,
                preprocessing_overrides=CHAMPION_PREP)
    print(f"\n{'='*65}")
    print(f"  Champion — ddi_alpha={ddi_alpha}  ({len(SEEDS)} seeds)")
    print(f"{'='*65}")
    for seed in SEEDS:
        run_name   = f"champion_full_alpha{atag}_seed{seed}"
        run_dir    = reports_dir / run_name
        legacy_dir = reports_dir / f"champion_full_seed{seed}"   # pre-DDI naming
        run_dir.mkdir(parents=True, exist_ok=True)
        already = list(run_dir.glob("result_*.json"))
        already_legacy = (ddi_alpha == 0.0) and list(legacy_dir.glob("result_*.json"))
        if already or already_legacy:
            label = run_name if already else f"champion_full_seed{seed} (legacy)"
            print(f"  [SKIP] {label} — already done")
            results_log.append(f"SKIP: {run_name}")
            continue
        log_path = run_dir / "training_log.txt"
        cmd = (f"python -u src/train.py --config {cfgp} "
               f"{BACKBONE_FLAGS} {LAB_FLAGS} "
               f"--ddi_alpha {ddi_alpha} "
               f"--seed {seed} --results_dir {run_dir}")
        print(f"\n  --- {run_name} ---")
        try:
            with open(log_path, "w") as lf:
                proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                        stderr=subprocess.STDOUT, text=True)
                for line in proc.stdout:
                    print(line, end="")
                    lf.write(line)
                proc.wait()
            status = "SUCCESS" if proc.returncode == 0 else f"FAILED ({proc.returncode})"
            results_log.append(f"{status}: {run_name}")
        except Exception as e:
            results_log.append(f"CRASH: {run_name}: {e}")
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f"Champion config ready.  reports_dir = {reports_dir}")
print(f"Seeds: {SEEDS}")
print(f"DDI alphas: {DDI_ALPHAS}  ->  {len(SEEDS) * len(DDI_ALPHAS)} total runs")
print()
print("  alpha=0.0  : baseline — no DDI loss, max Jaccard")
print("  alpha=0.2  : balanced — small Jaccard cost, meaningful DDI reduction")
print("  alpha=0.5  : safety-first — DDI rate below 0.06 target")


In [ ]:
# ── Smoke test: 1 epoch × alpha=0.0 before committing to full runs ────────────
Path("smoke_s16").mkdir(exist_ok=True)
cfg_path = "s16_smoke.yaml"
make_config(cfg_path,
            model_overrides=CHAMPION_MODEL,
            training_overrides=CHAMPION_TRAINING,
            preprocessing_overrides=CHAMPION_PREP,
            smoke_epochs=1)
cmd = (f"python -u src/train.py --config {cfg_path} "
       f"{BACKBONE_FLAGS} {LAB_FLAGS} "
       f"--ddi_alpha 0.0 "
       f"--seed 42 --results_dir smoke_s16/champion")
print(f"SMOKE: running 1 epoch...")
proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
for ln in (proc.stdout + proc.stderr).strip().split("\n")[-8:]: print(ln)
status = "PASS" if proc.returncode == 0 else f"FAIL ({proc.returncode})"
print(f">>> {status}")
if "FAIL" in status:
    raise RuntimeError("Smoke test failed — check config before running full seeds.")
print("Smoke passed.")


In [ ]:
# ── [1/3] alpha=0.0 — baseline, no DDI loss  (~5h) ───────────────────────────
# Max Jaccard configuration. Also checks for legacy champion_full_seed{s} results
# from before DDI supervision was added — those count as alpha=0.0 and are skipped.
run_champion(ddi_alpha=0.0)


In [ ]:
# ── [2/3] alpha=0.2 — balanced DDI supervision  (~5h) ────────────────────────
# Sweep 17 Gap C: -0.004 Jaccard, -16% DDI rate (0.082 -> 0.069).
# Best clinical trade-off: strong safety signal, minimal performance cost.
run_champion(ddi_alpha=0.2)


In [ ]:
# ── [3/3] alpha=0.5 — safety-first DDI supervision  (~5h) ────────────────────
# Sweep 17 Gap C: DDI rate drops to 0.047, below the 0.06 target.
# Jaccard cost: -0.018 vs alpha=0.0. Worth it if clinical safety is paramount.
run_champion(ddi_alpha=0.5)


In [ ]:
total   = len(results_log)
success = sum(1 for r in results_log if "SUCCESS" in r)
failed  = sum(1 for r in results_log if "FAILED"  in r)
crashed = sum(1 for r in results_log if "CRASH"   in r)
skipped = sum(1 for r in results_log if "SKIP"    in r)
print(f"\nRun log: {total} total | {success} success | {failed} failed | {crashed} crashed | {skipped} skipped")
if failed + crashed > 0:
    print("\nFailed/crashed runs:")
    for r in results_log:
        if "SUCCESS" not in r and "SKIP" not in r: print(f"  {r}")


In [ ]:
import json, numpy as np
from pathlib import Path

def get_metric(d, key):
    sec = d.get("test_metrics", {})
    for k in [key, key.replace("_", " "), key.title()]:
        if k in sec: return float(sec[k])
        if k in d:   return float(d[k])
    return 0.0

def load_alpha(ddi_alpha):
    # Load results for a given alpha — checks new AND legacy naming.
    atag = str(ddi_alpha).replace(".", "p")
    runs = []
    for seed in SEEDS:
        # New naming first
        for jp in sorted((reports_dir / f"champion_full_alpha{atag}_seed{seed}").glob("result_*.json")):
            with open(jp) as f: runs.append(json.load(f))
        # Legacy naming (alpha=0.0 only)
        if ddi_alpha == 0.0:
            for jp in sorted((reports_dir / f"champion_full_seed{seed}").glob("result_*.json")):
                with open(jp) as f: runs.append(json.load(f))
    # Deduplicate by seed
    seen, deduped = set(), []
    for d in runs:
        s = d.get("seed", id(d))
        if s not in seen:
            seen.add(s)
            deduped.append(d)
    return deduped

print("=" * 70)
print("  MIRROR — Champion × DDI Alpha Comparison  (Sweep 16, 5-seed official)")
print("=" * 70)
print(f"  {'alpha':<10} {'Jac':>7} {'±std':>6}  {'F1':>6}  {'DDI Rate':>9}  {'PRAUC':>7}  n  note")
print("  " + "-"*68)

best_jac, best_ddi = 0.0, 1.0
for alpha in DDI_ALPHAS:
    runs = load_alpha(alpha)
    if not runs:
        print(f"  alpha={alpha:<5}         (not yet run)")
        continue
    jac   = [get_metric(d, "jaccard")  for d in runs]
    f1    = [get_metric(d, "f1")       for d in runs]
    ddi   = [get_metric(d, "DDI Rate") for d in runs]
    prauc = [get_metric(d, "PRAUC")    for d in runs]
    std   = np.std(jac, ddof=1) if len(jac) > 1 else 0
    mean_jac = np.mean(jac)
    mean_ddi = np.mean(ddi)
    note = ""
    if alpha == 0.0: note = " <- max Jaccard"
    elif mean_ddi < 0.06: note = " <- DDI target met"
    elif mean_ddi < 0.075: note = " <- balanced"
    if mean_jac > best_jac: best_jac = mean_jac
    if mean_ddi < best_ddi: best_ddi = mean_ddi
    print(f"  alpha={alpha:<5}  {mean_jac:.4f} {std:.4f}  {np.mean(f1):.4f}  {mean_ddi:>9.4f}  {np.mean(prauc):.4f}  {len(jac)}{note}")

print()
print(f"  SafeDrug SOTA    : Jaccard 0.5021  DDI Rate ~0.0900")
print(f"  MIRROR (alpha=0) : best Jaccard {best_jac:.4f}  (+{best_jac - 0.5021:.4f} vs SOTA)")
print(f"  MIRROR (min DDI) : lowest DDI {best_ddi:.4f}")
print()
print("  DDI target: <= 0.06. 'alpha=0.2' = clinically aware champion.")
print("="*70)


In [ ]:
import zipfile
from pathlib import Path

zip_name = "sweep_16_champion_results.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(reports_dir.rglob("result_*.json")):
        zf.write(p, p.relative_to(reports_dir.parent))
    for p in sorted(reports_dir.rglob("training_log.txt")):
        zf.write(p, p.relative_to(reports_dir.parent))
print(f"Exported -> {zip_name}  ({sum(1 for _ in reports_dir.rglob('result_*.json'))} result files)")
